# Train Market Stream Models Across Architectures

Workflow:

1. Read C++-built CSV outputs: `market_stream_btcusdt.csv`, `features_btcusdt.csv`, and `train_dataset_btcusdt_h30.csv`
2. Use the existing feature columns and future-return labels without rebuilding them in Python
3. Create tabular and sequence datasets/loaders
4. Train sklearn, XGBoost, PyTorch MLP, LSTM, Transformer, and CNN models
5. Compare holdout results
6. Save the best model in a format `prediction_server.py` can serve

## 1. Setup

In [ ]:
from __future__ import annotations

import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ml_pipeline" / "common.py").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing ml_pipeline/common.py")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ML_DIR = REPO_ROOT / "ml_pipeline"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from common import FEATURE_COLUMNS
from lstm_model import CNNRegressor, LSTMRegressor, TransformerRegressor

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None
    print("xgboost is not installed. Run `pip install -r ml_pipeline/requirements.txt` to enable it.")

torch.set_num_threads(1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")
print(f"Feature columns: {FEATURE_COLUMNS}")

## 2. Config

In [ ]:
RAW_TICKS_CSV = REPO_ROOT / "data" / "market_stream_btcusdt.csv"
FEATURES_CSV = REPO_ROOT / "data" / "features_btcusdt.csv"
TRAIN_DATASET_CSV = REPO_ROOT / "data" / "train_dataset_btcusdt_h30.csv"
MODEL_OUT_DIR = REPO_ROOT / "models"

HORIZON_SEC = 30
TEST_SIZE = 0.20
RANDOM_STATE = 42
MAX_ROWS = None  # Example: 20_000 for faster experiments

SEQ_LEN = 60
BATCH_SIZE = 256
EPOCHS = 5
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.1
N_JOBS = 1

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Raw market stream: {RAW_TICKS_CSV}")
print(f"C++ features: {FEATURES_CSV}")
print(f"C++ training dataset: {TRAIN_DATASET_CSV}")

## 3. Read C++-Built Market Data, Features, And Labels

In [ ]:
raw_ticks = pd.read_csv(RAW_TICKS_CSV)
features_df = pd.read_csv(FEATURES_CSV)
dataset_df = pd.read_csv(TRAIN_DATASET_CSV)

required = set(FEATURE_COLUMNS + ["target_return", "price"])
missing = sorted(required - set(dataset_df.columns))
if missing:
    raise ValueError(f"Training dataset is missing required columns: {missing}")

dataset_df = dataset_df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLUMNS + ["target_return", "price"])
dataset_df = dataset_df.sort_values("ts").reset_index(drop=True) if "ts" in dataset_df.columns else dataset_df.reset_index(drop=True)
if MAX_ROWS is not None and len(dataset_df) > MAX_ROWS:
    dataset_df = dataset_df.iloc[-MAX_ROWS:].reset_index(drop=True)

display(raw_ticks.head())
display(features_df.head())
display(dataset_df.head())
display(dataset_df[FEATURE_COLUMNS + ["target_return"]].describe().T)
print(f"Raw market rows: {len(raw_ticks):,}")
print(f"Feature rows: {len(features_df):,}")
print(f"Training rows after cleanup: {len(dataset_df):,}")

## 4. Create Datasets And DataLoaders

In [ ]:
class TabularReturnDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


class SequenceReturnDataset(Dataset):
    def __init__(
        self,
        prices: np.ndarray,
        targets: np.ndarray,
        seq_len: int,
        ret_mean: float,
        ret_std: float,
    ) -> None:
        prices = np.asarray(prices, dtype=np.float64)
        log_returns = np.log(np.maximum(prices[1:], 1e-12) / np.maximum(prices[:-1], 1e-12))
        self.ret_mean = float(ret_mean)
        self.ret_std = float(ret_std if ret_std > 0 else 1.0)
        scaled = (log_returns - self.ret_mean) / self.ret_std

        X_seq = []
        y_seq = []
        for end_idx in range(seq_len, len(scaled)):
            target_idx = end_idx + 1
            if target_idx >= len(targets):
                break
            X_seq.append(scaled[end_idx - seq_len:end_idx].reshape(seq_len, 1))
            y_seq.append(targets[target_idx])

        self.X = torch.as_tensor(np.asarray(X_seq), dtype=torch.float32)
        self.y = torch.as_tensor(np.asarray(y_seq), dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


X = dataset_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y = dataset_df["target_return"].to_numpy(dtype=np.float32)
prices = dataset_df["price"].to_numpy(dtype=np.float64)

split_idx = int(len(dataset_df) * (1.0 - TEST_SIZE))
X_train_raw, X_test_raw = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

x_mean = X_train_raw.mean(axis=0)
x_std = X_train_raw.std(axis=0)
x_std = np.where(x_std > 0, x_std, 1.0)
X_train_scaled = (X_train_raw - x_mean) / x_std
X_test_scaled = (X_test_raw - x_mean) / x_std

tab_train_ds = TabularReturnDataset(X_train_scaled, y_train)
tab_test_ds = TabularReturnDataset(X_test_scaled, y_test)
tab_train_loader = DataLoader(tab_train_ds, batch_size=BATCH_SIZE, shuffle=False)
tab_test_loader = DataLoader(tab_test_ds, batch_size=BATCH_SIZE, shuffle=False)

train_log_returns = np.log(
    np.maximum(prices[1:split_idx], 1e-12) / np.maximum(prices[: split_idx - 1], 1e-12)
)
seq_ret_mean = float(train_log_returns.mean())
seq_ret_std = float(train_log_returns.std() if train_log_returns.std() > 0 else 1.0)
seq_all_ds = SequenceReturnDataset(prices, y, SEQ_LEN, seq_ret_mean, seq_ret_std)
seq_split_idx = int(len(seq_all_ds) * (1.0 - TEST_SIZE))
seq_train_ds = torch.utils.data.Subset(seq_all_ds, range(0, seq_split_idx))
seq_test_ds = torch.utils.data.Subset(seq_all_ds, range(seq_split_idx, len(seq_all_ds)))
seq_train_loader = DataLoader(seq_train_ds, batch_size=BATCH_SIZE, shuffle=False)
seq_test_loader = DataLoader(seq_test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Tabular train/test: {len(tab_train_ds):,} / {len(tab_test_ds):,}")
print(f"Sequence train/test: {len(seq_train_ds):,} / {len(seq_test_ds):,}")
print(f"Sequence normalization mean/std: {seq_all_ds.ret_mean:.8f} / {seq_all_ds.ret_std:.8f}")

## 5. Shared Metrics

In [ ]:
def regression_metrics(y_true: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred))),
        "mae": float(mean_absolute_error(y_true, pred)),
        "r2": float(r2_score(y_true, pred)),
        "direction_acc": float(np.mean(np.sign(y_true) == np.sign(pred))),
    }


results = []
artifacts = {}


def add_result(family: str, name: str, y_true: np.ndarray, pred: np.ndarray, seconds: float, artifact: dict) -> None:
    row = {"family": family, "architecture": name, "fit_seconds": float(seconds)}
    row.update(regression_metrics(y_true, pred))
    results.append(row)
    artifacts[name] = artifact
    print(f"{name}: RMSE={row['rmse']:.8f} MAE={row['mae']:.8f} DIR={row['direction_acc']:.3f}")

## 6. Train Sklearn And XGBoost

In [ ]:
sklearn_models = {
    "sklearn_ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    "sklearn_hgbt": HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=500,
        max_depth=6,
        min_samples_leaf=25,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    ),
    "sklearn_random_forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=10,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
    "sklearn_extra_trees": ExtraTreesRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=10,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
    "sklearn_mlp": make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(64, 32),
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE,
        ),
    ),
}

if XGBRegressor is not None:
    sklearn_models["xgboost_tree"] = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )

for name, model in sklearn_models.items():
    t0 = time.perf_counter()
    fitted = clone(model)
    fitted.fit(X_train_raw, y_train)
    pred = fitted.predict(X_test_raw)
    add_result(
        family="tabular_joblib",
        name=name,
        y_true=y_test,
        pred=pred,
        seconds=time.perf_counter() - t0,
        artifact={"format": "joblib", "model": fitted},
    )

## 7. Train PyTorch Tabular MLP

In [ ]:
class TorchMLPRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int = 64, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def train_torch_model(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, epochs: int) -> tuple[np.ndarray, np.ndarray, list[dict[str, float]], float]:
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    history = []
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        valid_losses = []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                valid_losses.append(float(loss_fn(model(xb), yb).detach().cpu()))
        history.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), "valid_loss": float(np.mean(valid_losses))})
        print(f"epoch={epoch:02d} train={history[-1]['train_loss']:.10f} valid={history[-1]['valid_loss']:.10f}")

    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for xb, yb in test_loader:
            pred = model(xb.to(DEVICE)).detach().cpu().numpy()
            y_pred.append(pred)
            y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred), history, time.perf_counter() - t0


torch_mlp = TorchMLPRegressor(len(FEATURE_COLUMNS), HIDDEN_SIZE, DROPOUT)
mlp_true, mlp_pred, mlp_history, mlp_seconds = train_torch_model(torch_mlp, tab_train_loader, tab_test_loader, EPOCHS)
add_result(
    family="torch_tabular",
    name="torch_mlp",
    y_true=mlp_true,
    pred=mlp_pred,
    seconds=mlp_seconds,
    artifact={"format": "torch_tabular", "model": torch_mlp.cpu(), "history": mlp_history},
)

## 8. Train PyTorch Sequence Models: LSTM, Transformer, CNN

These use normalized one-step log-return sequences because the prediction server builds the same sequence from incoming price points for `.pt` artifacts.

In [ ]:
sequence_models = {
    "lstm": LSTMRegressor(input_dim=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT),
    "transformer": TransformerRegressor(input_dim=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT, nhead=4),
    "cnn": CNNRegressor(input_dim=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT),
}

for name, model in sequence_models.items():
    print(f"Training {name}...")
    y_true_seq, y_pred_seq, history, seconds = train_torch_model(model, seq_train_loader, seq_test_loader, EPOCHS)
    add_result(
        family="torch_sequence_pt",
        name=name,
        y_true=y_true_seq,
        pred=y_pred_seq,
        seconds=seconds,
        artifact={"format": "pt", "model": model.cpu(), "history": history},
    )

## 9. Compare Results

In [ ]:
results_df = pd.DataFrame(results).sort_values(["rmse", "mae"]).reset_index(drop=True)
display(results_df)
server_ready_names = {name for name, artifact in artifacts.items() if artifact["format"] in {"joblib", "pt"}}
server_results_df = results_df[results_df["architecture"].isin(server_ready_names)].reset_index(drop=True)
best_name = str(results_df.iloc[0]["architecture"])
best_family = str(results_df.iloc[0]["family"])
best_server_name = str(server_results_df.iloc[0]["architecture"])
best_server_family = str(server_results_df.iloc[0]["family"])
print(f"Best overall model: {best_name} ({best_family})")
print(f"Best server-ready model: {best_server_name} ({best_server_family})")

## 10. Save Model For Server Inference

- sklearn and XGBoost artifacts save as `.joblib` and are served through the existing tabular feature path.
- LSTM, Transformer, and CNN artifacts save as `.pt` and are served through the sequence path in `prediction_server.py`.

In [ ]:
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
best_artifact = artifacts[best_server_name]
best_row = server_results_df.iloc[0].to_dict()

if best_artifact["format"] == "joblib":
    model_path = MODEL_OUT_DIR / f"pulse_{best_server_name}.joblib"
    bundle = {
        "model_type": best_server_name,
        "model_name": f"Pulse{best_server_name.replace('_', ' ').title().replace(' ', '')}",
        "horizon_sec": int(HORIZON_SEC),
        "feature_columns": FEATURE_COLUMNS,
        "model": best_artifact["model"],
        "metrics": best_row,
        "leaderboard": results_df.to_dict(orient="records"),
    }
    joblib.dump(bundle, model_path)
elif best_artifact["format"] == "pt":
    model_path = MODEL_OUT_DIR / f"pulse_{best_server_name}.pt"
    torch.save(
        {
            "model_type": best_server_name,
            "model_name": f"Pulse{best_server_name.title()}V1",
            "horizon_sec": int(HORIZON_SEC),
            "seq_len": int(SEQ_LEN),
            "input_dim": 1,
            "hidden_size": int(HIDDEN_SIZE),
            "num_layers": int(NUM_LAYERS),
            "dropout": float(DROPOUT),
            "nhead": 4,
            "ret_mean": float(seq_all_ds.ret_mean),
            "ret_std": float(seq_all_ds.ret_std),
            "state_dict": best_artifact["model"].state_dict(),
            "metrics": best_row,
            "leaderboard": results_df.to_dict(orient="records"),
        },
        model_path,
    )
else:
    raise ValueError(f"Best artifact format is not server-ready: {best_artifact['format']}")

print(f"Saved server artifact: {model_path}")
print("Run server:")
print(f"cd {ML_DIR} && python prediction_server.py --model ../models/{model_path.name} --host 127.0.0.1 --port 50061")